<a href="https://colab.research.google.com/github/peremartra/CH13/blob/main/CH12_NB02_arithmetic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CH12 · NB02 — Padding and KV cache arithmetic

Companion to section 12.7. Pure Python, no GPU required.

In [1]:
def padding_report(target_width, multiple=128):
    served = ((target_width + multiple - 1) // multiple) * multiple   #A
    return served, served - target_width

def snap_to_aligned(target_width, multiple=128):
    return (target_width // multiple) * multiple                      #B

raw = int(8192 * 0.80)                 # 6553, the naive 20% cut
served, waste = padding_report(raw)    # kernel actually runs 6656 wide
aligned = snap_to_aligned(raw)         # prune to 6528 instead, zero waste
print(f"naive target {raw}: served {served}, wasted columns {waste}")
print(f"aligned target {aligned}: served {aligned}, wasted columns 0")

naive target 6553: served 6656, wasted columns 103
aligned target 6528: served 6528, wasted columns 0


In [2]:
def kv_cache_bytes(L_attn, H_kv, d_head, seq_len, batch, dtype_bytes=2):
    return 2 * L_attn * H_kv * d_head * seq_len * batch * dtype_bytes

base = kv_cache_bytes(L_attn=28, H_kv=8, d_head=128, seq_len=2048, batch=64)
slim = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=2048, batch=64)
print(f"cache after pruning is {slim / base:.1%} of the original")

cache after pruning is 57.1% of the original


In [3]:
small = kv_cache_bytes(L_attn=16, H_kv=8, d_head=128, seq_len=300, batch=8)
print(f"{small / 1e6:.2f} MB ({small / 2**20:.0f} MiB)")

157.29 MB (150 MiB)
